# protosearch — Protein Family Survey

Generic template for surveying any protein family across a user-supplied proteome.

**Pipeline:** HMMER retrieval → ESM2 embedding → t-SNE clustering → IQ-TREE phylogeny → Ancestral reconstruction → Cluster labeling

---
**To use:** Fill in cells marked `USER INPUT` and run top to bottom.

For a worked example with AGPAT acyltransferases see `agpat_crustacea.ipynb` in the agpat/ directory.

In [ ]:
# [00] Clone repo and configure path
import subprocess, sys
subprocess.run(['git', 'clone', 'https://github.com/bjdraper/protosearch.git'], check=True)
sys.path.insert(0, 'protosearch')

In [ ]:
# [02] Install dependencies
!apt-get install -qq hmmer mafft fasttree
!pip install -q fair-esm faiss-cpu leidenalg python-igraph openTSNE biopython logomaker pyyaml tqdm seaborn

In [ ]:
# [03] Mount Google Drive and set survey name
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

# ── USER INPUT ────────────────────────────────────────────────────────────────
SURVEY_NAME = 'my_protein_family'   # e.g. 'kinase_insects', 'agpat_crustacea'
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = Path(f'/content/drive/MyDrive/{SURVEY_NAME}')
DATA_DIR     = PROJECT_ROOT / 'data'
RESULTS_DIR  = PROJECT_ROOT / 'results'
for d in [DATA_DIR, RESULTS_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

In [ ]:
# [04] Define protein family  ── USER INPUT ────────────────────────────────────

# Replace with your family's reference sequences (UniProt accessions)
REFERENCE_PROBES = [
    # {'id': 'MYPROTEIN_HUMAN', 'acc': 'P12345', 'label': 'MyProtein', 'colour': '#D94040'},
]

# Pfam domain ID(s) for HMMER prefilter (e.g. 'PF01553' for acyltransferases)
PFAM_IDS = ['PF00000']   # replace with your Pfam ID

# Protein length filter (adjust to your family)
MIN_LEN = 200
MAX_LEN = 500

# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# [05] Write config.yaml
import yaml
config = {
    'paths': {'data_dir': str(DATA_DIR), 'results_dir': str(RESULTS_DIR)},
    'reference_probes': REFERENCE_PROBES,
    'pfam_ids': PFAM_IDS,
    'length_filter': {'min': MIN_LEN, 'max': MAX_LEN},
    'embedding': {'model': 'esm2_t33_650M_UR50D', 'layer': 33, 'batch_size': 32},
    'clustering': {'leiden_resolution': 2.0, 'tsne_perplexity': 100},
    'tree': {'aligner': 'mafft', 'model': 'LG+G4'},
}
config_path = PROJECT_ROOT / 'config.yaml'
with open(config_path, 'w') as f: yaml.dump(config, f)
print('Config written to', config_path)

In [ ]:
# [06] Download reference probes from UniProt
from protosearch import utils
probe_fasta = DATA_DIR / 'reference_probes.faa'
utils.fetch_uniprot_sequences([p['acc'] for p in REFERENCE_PROBES], probe_fasta)
print(f'Fetched {len(REFERENCE_PROBES)} reference probes → {probe_fasta}')

In [ ]:
# [07] Download Pfam HMM profile(s)
from protosearch import search
hmm_dir = DATA_DIR / 'hmm_profiles'
hmm_dir.mkdir(exist_ok=True)
for pfam_id in PFAM_IDS:
    search.download_pfam_hmm(pfam_id, hmm_dir)
print('HMM profiles ready in', hmm_dir)

In [ ]:
# [08] Scan Drive for proteome FASTA files  ── USER INPUT ─────────────────────
import os
proteins_raw_dir = DATA_DIR / 'proteins_raw'
found = sorted(proteins_raw_dir.glob('*.faa')) + sorted(proteins_raw_dir.glob('*.fasta'))
print('Found FASTA files:')
for f in found: print(' ', f.name)

# Set INPUT_FASTAS to the files you want to include
INPUT_FASTAS = found   # or list a subset: [found[0], found[2]]
# ─────────────────────────────────────────────────────────────────────────────

raw_fasta = DATA_DIR / 'proteins_raw_combined.faa'
with open(raw_fasta, 'w') as out:
    for f in INPUT_FASTAS:
        out.write(open(f).read())
print(f'Combined {len(INPUT_FASTAS)} file(s) → {raw_fasta}')

In [ ]:
# [09] Filter proteins by length + dedup
from protosearch import utils
filtered_fasta = DATA_DIR / 'proteins_filtered.faa'
n = utils.filter_and_dedup(raw_fasta, filtered_fasta, min_len=MIN_LEN, max_len=MAX_LEN)
print(f'{n} proteins after length filter ({MIN_LEN}–{MAX_LEN} AA) → {filtered_fasta}')

In [ ]:
# [10] HMMER prefilter
from protosearch import search
hmmer_hits_fasta = DATA_DIR / 'hmmer_hits.faa'
hmmer_hits_tsv   = RESULTS_DIR / 'hmmer_hits.tsv'
search.run_hmmer(filtered_fasta, hmm_dir, hmmer_hits_fasta, hmmer_hits_tsv)
print(f'HMMER hits → {hmmer_hits_fasta}')

In [ ]:
# [11] ESM2 embedding of HMMER hits
from protosearch import embed
hits_emb_path = DATA_DIR / 'embeddings' / 'hmmer_hits.npy'
hits_ids_path = DATA_DIR / 'embeddings' / 'hmmer_hits_ids.txt'
embed.embed_fasta(hmmer_hits_fasta, hits_emb_path, hits_ids_path, device='cuda')
print('Embeddings saved.')

In [ ]:
# [12] ESM2 embedding of reference probes
ref_emb_path = DATA_DIR / 'embeddings' / 'ref_embeddings.npy'
ref_ids_path = DATA_DIR / 'embeddings' / 'ref_embeddings_ids.txt'
embed.embed_fasta(probe_fasta, ref_emb_path, ref_ids_path, device='cuda')
print('Reference embeddings saved.')

In [ ]:
# [16] Load embeddings for clustering
import numpy as np
emb  = np.load(hits_emb_path)
ids  = open(hits_ids_path).read().splitlines()
ref_emb = np.load(ref_emb_path)
ref_ids = open(ref_ids_path).read().splitlines()
print(f'Hits: {emb.shape}, Refs: {ref_emb.shape}')

In [ ]:
# [17] Leiden clustering
from protosearch import cluster
labels = cluster.leiden_cluster(emb, resolution=2.0)
print(f'{len(set(labels))} clusters found')

In [ ]:
# [18] t-SNE plot
from protosearch import cluster, visualize
tsne_coords = cluster.run_tsne(emb, perplexity=100)
visualize.plot_tsne(tsne_coords, labels, ids, ref_emb, ref_ids,
                    REFERENCE_PROBES, RESULTS_DIR / 'tsne.png')

In [ ]:
# [20] MAFFT + FastTree per cluster
from protosearch import tree, utils
for clu in sorted(set(labels)):
    clu_ids = [i for i, l in zip(ids, labels) if l == clu]
    clu_fasta = RESULTS_DIR / f'cluster_{clu}_hits.faa'
    utils.extract_sequences(hmmer_hits_fasta, clu_ids, clu_fasta)
    tree.align_and_tree(clu_fasta, RESULTS_DIR / f'cluster_{clu}')
print('Trees built.')

In [ ]:
# [21] IQ-TREE2 ancestral state reconstruction
from protosearch import tree
for clu in sorted(set(labels)):
    aligned = RESULTS_DIR / f'cluster_{clu}_aligned.faa'
    if aligned.exists():
        tree.run_iqtree_asr(aligned, RESULTS_DIR / f'cluster_{clu}_iqtree')
print('ASR complete.')

In [ ]:
# [22] Parse ASR results + generate sequence logos
from protosearch import asr, visualize
for clu in sorted(set(labels)):
    state_file = RESULTS_DIR / f'cluster_{clu}_iqtree' / 'alignment.state'
    if state_file.exists():
        anc = asr.parse_state_file(state_file)
        visualize.plot_ancestral_logo(anc, RESULTS_DIR / f'cluster_{clu}_ancestral_logo.png')
print('Done.')